# LeetCode 2268 — Minimum Number of Keypresses bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-keypresses/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Greedy, Sorting, Counting

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def minimumKeypresses(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài "tự thiết kế bàn phím" thành bài **gán chi phí**: có 9 ô chi phí 1, 9 ô chi phí 2, 8 ô chi phí 3 (đủ cho 26 chữ); mỗi chữ chiếm một ô; tổng chi phí = Σ (tần suất × chi phí ô).
2. Chứng minh tham lam "**chữ xuất hiện nhiều nhất nhận ô rẻ nhất**" bằng **bất đẳng thức sắp xếp lại** (rearrangement inequality) / lập luận đổi chỗ.
3. Cài đặt: vét cạn hoán vị (kiểm chứng), sắp xếp tần suất, đếm phân phối, heap, công thức `i // 9 + 1`.
4. Tổng quát hóa cho `k` nút, mỗi nút chứa tối đa `c` chữ, và chi phí vị trí tùy ý (không giảm).
5. Tránh các bẫy: sắp xếp tăng dần, dùng thứ tự bảng chữ cái, lệch công thức chi phí.

## 2. Đề bài & ví dụ

Có bàn phím 9 nút (đánh số 1..9). Bạn **tự chọn** chữ cái cho mỗi nút sao cho:
- Mọi chữ cái thường (26 chữ) đều được gán;
- Mỗi chữ thuộc **đúng một** nút;
- Mỗi nút có **tối đa 3** chữ.

Gõ chữ thứ `t` (1-based) trên một nút cần bấm nút đó `t` lần. Cho chuỗi `s`, trả về **số lần bấm ít nhất** để gõ `s` (với bàn phím do bạn thiết kế tối ưu).

**Ví dụ 1:** `s = "apple"` → `5`: 4 chữ khác nhau `a, p, l, e` đều đặt ở vị trí đầu của 4 nút ⇒ mỗi chữ 1 lần bấm.

**Ví dụ 2:** `s = "abcdefghijkl"` → `15`: 9 chữ đầu ở vị trí 1 (9 lần bấm), 3 chữ còn lại ở vị trí 2 (6 lần bấm).

**Ràng buộc:** `1 ≤ len(s) ≤ 10⁵`, `s` gồm chữ cái thường.

In [1]:
from collections import Counter
from itertools import permutations
import heapq, random, time, string

examples = [("apple", 5), ("abcdefghijkl", 15)]
for s, exp in examples:
    print(s, "-> tần suất", Counter(s).most_common(), "| kỳ vọng", exp)

apple -> tần suất [('p', 2), ('a', 1), ('l', 1), ('e', 1)] | kỳ vọng 5
abcdefghijkl -> tần suất [('a', 1), ('b', 1), ('c', 1), ('d', 1), ('e', 1), ('f', 1), ('g', 1), ('h', 1), ('i', 1), ('j', 1), ('k', 1), ('l', 1)] | kỳ vọng 15


## 3. Kiến thức nền

### 3.1 Tần suất ký tự với `Counter` và sắp xếp giảm dần

In [2]:
cnt = Counter("mississippi")
print(cnt, sorted(cnt.values(), reverse=True), cnt.most_common(2))

Counter({'i': 4, 's': 4, 'p': 2, 'm': 1}) [4, 4, 2, 1] [('i', 4), ('s', 4)]


### 3.2 Chi phí của "ô" thứ `i` (0-based) khi xếp ô rẻ trước

Có 9 nút; xếp các ô theo thứ tự: 9 ô vị trí 1, rồi 9 ô vị trí 2, rồi 9 ô vị trí 3 ⇒ ô thứ `i` có chi phí `i // 9 + 1`.

In [3]:
print([i // 9 + 1 for i in range(26)])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3]


### 3.3 Bất đẳng thức sắp xếp lại (rearrangement inequality)

Cho hai dãy `a₁ ≥ a₂ ≥ … ≥ aₙ` và `b₁ ≤ b₂ ≤ … ≤ bₙ`. Trong mọi cách ghép `a` với một hoán vị của `b`, tổng `Σ aᵢ · b_{σ(i)}` **nhỏ nhất** khi ghép lớn với nhỏ (`a₁` với `b₁`, …). Đây chính là "tần suất lớn ghép chi phí nhỏ".

In [4]:
a = [5, 3, 1]; b = [1, 2, 3]
for perm in permutations(b):
    print(perm, sum(x * y for x, y in zip(a, perm)))

(1, 2, 3) 14
(1, 3, 2) 16
(2, 1, 3) 16
(2, 3, 1) 20
(3, 1, 2) 20
(3, 2, 1) 22


### 3.4 `heapq` để lấy lần lượt phần tử lớn nhất

Đẩy `-tần_suất` vào heap (min-heap) để pop ra tần suất lớn nhất trước — một cách khác thay cho sắp xếp.

In [5]:
h = [-v for v in Counter("apple").values()]; heapq.heapify(h)
print([-heapq.heappop(h) for _ in range(len(h))])

[2, 1, 1, 1]


### 3.5 Có bao nhiêu bàn phím khác nhau?

Số cách chia 26 chữ vào 9 nút (mỗi nút ≤ 3 chữ, có thứ tự trong nút) là khổng lồ — vét cạn bàn phím là vô vọng; ta phải suy luận cấu trúc. Ước lượng thô: riêng số cách xếp 26 chữ vào 27 ô đã là `27!/1! ≈ 1,1·10²⁸`.

In [6]:
from math import factorial, perm
print(f"số cách xếp 26 chữ vào 27 ô: {perm(27, 26):.3e}")
print(f"nhưng chỉ cần biết mỗi chữ thuộc mức chi phí nào: tối đa 3^26 = {3**26:.3e} khả năng — vẫn quá lớn")

số cách xếp 26 chữ vào 27 ô: 1.089e+28
nhưng chỉ cần biết mỗi chữ thuộc mức chi phí nào: tối đa 3^26 = 2.542e+12 khả năng — vẫn quá lớn


## 4. Ý tưởng nền tảng / trực giác

Thiết kế bàn phím = phân phối 26 chữ vào 27 "ô" (9 nút × 3 vị trí), ô ở vị trí `t` tốn `t` lần bấm. Chi phí gõ `s` = Σ_chữ (số lần chữ xuất hiện) × (chi phí ô của nó). Chữ không xuất hiện thì đặt đâu cũng được (chi phí 0).

Trực giác: chữ **hay dùng** nên ở ô **rẻ**. Với `s = "abcdefghijkl"` (12 chữ, mỗi chữ 1 lần): 9 chữ bất kỳ vào 9 ô chi phí 1, 3 chữ còn lại vào ô chi phí 2 ⇒ `9 + 6 = 15`.

Tính tay `s = "aaaabbbcc" + "defghijkl"` (a:4, b:3, c:2, 9 chữ khác 1 lần):

| hạng | chữ | tần suất | chi phí ô | đóng góp |
|---|---|---|---|---|
| 1–9 | a, b, c, d, e, f, g, h, i | 4, 3, 2, 1×6 | 1 | 4 + 3 + 2 + 6 = 15 |
| 10–12 | j, k, l | 1, 1, 1 | 2 | 6 |

Tổng 21. Nếu lỡ đặt `a` ở vị trí 2 và `j` ở vị trí 1 thì tăng `4·2 + 1 − (4 + 2) = 3`.

In [7]:
s = "aaaabbbcc" + "defghijkl"
vals = sorted(Counter(s).values(), reverse=True)
print("tần suất giảm dần:", vals)
print("chi phí ô         :", [i // 9 + 1 for i in range(len(vals))])
print("tổng =", sum(v * (i // 9 + 1) for i, v in enumerate(vals)))

tần suất giảm dần: [4, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1]
chi phí ô         : [1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2]
tổng = 21


## 5. Cách 1 — Brute Force / baseline: thử mọi cách gán chữ (đã xuất hiện) vào các ô

**Ý tưởng:** chỉ các chữ **có xuất hiện** mới ảnh hưởng chi phí. Với `d` chữ khác nhau, thử mọi cách chọn và sắp `d` ô trong 27 ô (theo chi phí) — tương đương thử mọi hoán vị gán `d` chữ vào danh sách chi phí ô. Chỉ khả thi với `d` nhỏ.

In [8]:
SLOT_COST = [t for t in (1, 2, 3) for _ in range(9)]        # 27 ô: 9 ô mỗi mức chi phí

def brute(s):
    freq = list(Counter(s).values()); d = len(freq)
    best = None
    # chọn d ô (theo chỉ số) và gán theo mọi thứ tự = hoán vị chập d của 27 ô
    for slots in permutations(range(27), d):
        cost = sum(f * SLOT_COST[j] for f, j in zip(freq, slots))
        if best is None or cost < best: best = cost
    return best

for s, exp in [("apple", 5), ("aab", 3)]:
    got = brute(s)
    print(s, got, "OK" if got == exp else "SAI")

apple 5 OK
aab 3 OK


**Độ phức tạp:** `27!/(27 − d)!` hoán vị — `d = 4` đã ~4,2·10⁵; `d = 26` là ~10²⁸. Không khả thi ngoài `d` rất nhỏ.

**Nút thắt:** hầu hết các cách gán là "rõ ràng tệ" (chữ hay dùng ở ô đắt). Bất đẳng thức sắp xếp lại cho biết **ngay** cách gán tối ưu.

### 5.1 Baseline tốt hơn: vét cạn trên **chi phí** thay vì trên ô

Vì các ô cùng chi phí là như nhau, chỉ cần quyết định mỗi chữ nhận chi phí 1, 2 hay 3, với ràng buộc ≤ 9 chữ mỗi mức: `3^d` khả năng.

In [9]:
from itertools import product
def brute_levels(s):
    freq = list(Counter(s).values()); best = None
    for levels in product((1, 2, 3), repeat=len(freq)):
        if all(levels.count(t) <= 9 for t in (1, 2, 3)):
            c = sum(f * t for f, t in zip(freq, levels))
            if best is None or c < best: best = c
    return best

for s, exp in examples:
    if len(set(s)) <= 12:
        got = brute_levels(s); print(s, got, "OK" if got == exp else "SAI")

apple 5 OK


abcdefghijkl 15 OK


`3^12 ≈ 5,3·10⁵` — vẫn mũ, chỉ dùng để kiểm chứng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** sắp xếp tần suất **giảm dần** `f₁ ≥ f₂ ≥ … ≥ f_d`; gán `f_i` (1-based) vào ô chi phí `⌈i/9⌉` = `(i − 1)//9 + 1`. Đáp án `Σ f_i · ((i − 1)//9 + 1)`.

**Lập luận đổi chỗ:** giả sử một cách gán tối ưu có chữ `x` (tần suất `fx`) ở ô chi phí `cx` và chữ `y` (tần suất `fy`) ở ô chi phí `cy` với `fx > fy` nhưng `cx > cy`. Đổi chỗ hai chữ: chi phí thay đổi `fx·cy + fy·cx − fx·cx − fy·cy = (fx − fy)(cy − cx) < 0` — giảm hẳn, mâu thuẫn tính tối ưu. Vậy trong cách gán tối ưu, tần suất lớn hơn luôn ở ô không đắt hơn ⇒ đúng là cách tham lam (sai khác chỉ ở các chữ cùng tần suất, không ảnh hưởng tổng).

**Quy tắc:**
- **Nếu** chữ đứng hạng `1..9` **thì** chi phí 1; hạng `10..18` ⇒ 2; hạng `19..26` ⇒ 3.
- Chữ không xuất hiện không cần xét.

**Ràng buộc "mỗi chữ đúng một nút" và "26 chữ đều được gán"** luôn thỏa: 26 ≤ 27 ô, các chữ không xuất hiện lấp các ô còn lại.

**Về các chữ cùng tần suất:** thứ tự giữa chúng không ảnh hưởng tổng (đổi chỗ hai chữ cùng tần suất làm thay đổi `(fx − fy)(…) = 0`). Vì vậy không cần quy tắc phá hòa — khác với các bài yêu cầu xuất *chính* bàn phím theo thứ tự từ điển.

### 6.1 Minh họa lập luận đổi chỗ bằng số

Lấy một bàn phím "tệ" (chữ hay dùng ở vị trí sâu), lặp: tìm cặp vi phạm (tần suất lớn hơn nhưng chi phí cao hơn), đổi chỗ, in chi phí — chi phí giảm dần cho tới khi bằng đáp án tham lam.

In [10]:
s_demo = "e" * 9 + "t" * 7 + "a" * 5 + "o" * 4 + "abcdefghijklm"
freq = Counter(s_demo)
letters = sorted(freq)                              # thứ tự bảng chữ cái - một bàn phím tùy ý
cost_of = {ch: i // 9 + 1 for i, ch in enumerate(sorted(letters, key=lambda c: freq[c]))}   # cố tình đảo: ít dùng -> rẻ
def total(cm): return sum(freq[c] * cm[c] for c in freq)
print("ban đầu:", total(cost_of))
changed = True
while changed:
    changed = False
    for x in letters:
        for y in letters:
            if freq[x] > freq[y] and cost_of[x] > cost_of[y]:
                cost_of[x], cost_of[y] = cost_of[y], cost_of[x]; changed = True
                print(f"đổi '{x}'(x{freq[x]}) <-> '{y}'(x{freq[y]}) -> chi phí {total(cost_of)}")
                break
        if changed: break
greedy_val = sum(v * (i // 9 + 1) for i, v in enumerate(sorted(freq.values(), reverse=True)))
print("cuối cùng:", total(cost_of), "| công thức tham lam:", greedy_val)

ban đầu: 67
đổi 'a'(x6) <-> 'b'(x1) -> chi phí 62
đổi 'e'(x10) <-> 'a'(x6) -> chi phí 58
đổi 'a'(x6) <-> 'c'(x1) -> chi phí 53
đổi 'o'(x4) <-> 'd'(x1) -> chi phí 50
đổi 't'(x7) <-> 'a'(x6) -> chi phí 49
đổi 'a'(x6) <-> 'f'(x1) -> chi phí 44
cuối cùng: 44 | công thức tham lam: 44


Mỗi lần đổi chỗ giảm đúng `(fx − fy)(cx − cy) > 0`; quá trình dừng ở một bàn phím không còn cặp vi phạm — chính là bàn phím tham lam (sai khác chỉ ở các chữ cùng tần suất).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp tần suất giảm dần + công thức (cách doocs)

In [11]:
def sort_greedy(s):
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(v * (i // 9 + 1) for i, v in enumerate(vals))

for s, exp in examples:
    got = sort_greedy(s)
    print(got, "OK" if got == exp else "SAI")

5 OK
15 OK


**Độ phức tạp cách 2:** O(n) đếm + O(26 log 26) sắp xếp ⇒ **O(n)**; bộ nhớ O(26).

### 7.2 Cách 3 — Mảng đếm 26 phần tử (không dùng `Counter`)

In [12]:
def count_array(s):
    cnt = [0] * 26
    for ch in s: cnt[ord(ch) - 97] += 1
    cnt.sort(reverse=True)
    total = 0
    for i, v in enumerate(cnt):
        if v == 0: break
        total += v * (i // 9 + 1)
    return total

for s, exp in examples:
    print(count_array(s), "OK" if count_array(s) == exp else "SAI")

5 OK
15 OK


### 7.3 Cách 4 — Heap lấy dần tần suất lớn nhất

In [13]:
def heap_greedy(s):
    h = [-v for v in Counter(s).values()]; heapq.heapify(h)
    total = 0; i = 0
    while h:
        total += -heapq.heappop(h) * (i // 9 + 1); i += 1
    return total

for s, exp in examples:
    print(heap_greedy(s), "OK" if heap_greedy(s) == exp else "SAI")

5 OK
15 OK


Heap tiện khi tần suất **đến dần** (streaming) hoặc khi chỉ cần một phần đầu; ở đây sắp xếp 26 phần tử đã đủ.

### 7.4 Cách 5 — Gom theo khối 9 bằng lát cắt

In [14]:
def by_blocks(s):
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(cost * sum(vals[9 * (cost - 1): 9 * cost]) for cost in (1, 2, 3))

for s, exp in examples:
    print(by_blocks(s), "OK" if by_blocks(s) == exp else "SAI")

5 OK
15 OK


Viết theo "khối 9 chữ đắt dần" — đọc lên rất sát cách bố trí bàn phím thật.

### 7.5 Cách 6 — Dựng **bàn phím cụ thể** và đếm thật số lần bấm

Không chỉ trả số, mà dựng `keypad[nút] = [chữ vị trí 1, chữ vị trí 2, chữ vị trí 3]` theo tham lam, rồi gõ `s` để đếm — cách kiểm chứng "đầu cuối" (end-to-end).

In [15]:
def build_keypad(s):
    order = [ch for ch, _ in Counter(s).most_common()]
    order += [ch for ch in string.ascii_lowercase if ch not in order]      # chữ không dùng lấp chỗ trống
    keypad = [[] for _ in range(9)]
    for i, ch in enumerate(order):
        keypad[i % 9].append(ch)                  # vòng 1: vị trí 1 của 9 nút, vòng 2: vị trí 2, ...
    return keypad

def type_cost(keypad, s):
    pos = {ch: t + 1 for btn in keypad for t, ch in enumerate(btn)}
    return sum(pos[ch] for ch in s)

kp = build_keypad("abcdefghijkl")
print(kp)
print("số lần bấm:", type_cost(kp, "abcdefghijkl"), "| mỗi nút ≤ 3 chữ:", all(len(b) <= 3 for b in kp),
      "| đủ 26 chữ:", sorted(c for b in kp for c in b) == list(string.ascii_lowercase))

[['a', 'j', 's'], ['b', 'k', 't'], ['c', 'l', 'u'], ['d', 'm', 'v'], ['e', 'n', 'w'], ['f', 'o', 'x'], ['g', 'p', 'y'], ['h', 'q', 'z'], ['i', 'r']]
số lần bấm: 15 | mỗi nút ≤ 3 chữ: True | đủ 26 chữ: True


## 8. TRACE từng bước

Trace cách 2 trên `s = "thequickbrownfoxjumpsoverthelazydog"` (pangram — đủ 26 chữ).

In [16]:
def trace_greedy(s):
    items = sorted(Counter(s).items(), key=lambda kv: (-kv[1], kv[0]))
    total = 0
    for i, (ch, v) in enumerate(items):
        cost = i // 9 + 1; total += v * cost
        print(f"hạng {i + 1:>2}: '{ch}' x{v:<2} -> ô chi phí {cost}, đóng góp {v * cost:<2}, tổng {total}")
    return total

print("Kết quả:", trace_greedy("thequickbrownfoxjumpsoverthelazydog"))

hạng  1: 'o' x4  -> ô chi phí 1, đóng góp 4 , tổng 4
hạng  2: 'e' x3  -> ô chi phí 1, đóng góp 3 , tổng 7
hạng  3: 'h' x2  -> ô chi phí 1, đóng góp 2 , tổng 9
hạng  4: 'r' x2  -> ô chi phí 1, đóng góp 2 , tổng 11
hạng  5: 't' x2  -> ô chi phí 1, đóng góp 2 , tổng 13
hạng  6: 'u' x2  -> ô chi phí 1, đóng góp 2 , tổng 15
hạng  7: 'a' x1  -> ô chi phí 1, đóng góp 1 , tổng 16
hạng  8: 'b' x1  -> ô chi phí 1, đóng góp 1 , tổng 17
hạng  9: 'c' x1  -> ô chi phí 1, đóng góp 1 , tổng 18
hạng 10: 'd' x1  -> ô chi phí 2, đóng góp 2 , tổng 20
hạng 11: 'f' x1  -> ô chi phí 2, đóng góp 2 , tổng 22
hạng 12: 'g' x1  -> ô chi phí 2, đóng góp 2 , tổng 24
hạng 13: 'i' x1  -> ô chi phí 2, đóng góp 2 , tổng 26
hạng 14: 'j' x1  -> ô chi phí 2, đóng góp 2 , tổng 28
hạng 15: 'k' x1  -> ô chi phí 2, đóng góp 2 , tổng 30
hạng 16: 'l' x1  -> ô chi phí 2, đóng góp 2 , tổng 32
hạng 17: 'm' x1  -> ô chi phí 2, đóng góp 2 , tổng 34
hạng 18: 'n' x1  -> ô chi phí 2, đóng góp 2 , tổng 36
hạng 19: 'p' x1  -> ô chi phí 3

Trace ví dụ 1 `"apple"`: chỉ 4 chữ khác nhau ⇒ tất cả ở hạng 1–4, chi phí 1 ⇒ tổng = độ dài chuỗi = 5.

In [17]:
print("Kết quả:", trace_greedy("apple"))

hạng  1: 'p' x2  -> ô chi phí 1, đóng góp 2 , tổng 2
hạng  2: 'a' x1  -> ô chi phí 1, đóng góp 1 , tổng 3
hạng  3: 'e' x1  -> ô chi phí 1, đóng góp 1 , tổng 4
hạng  4: 'l' x1  -> ô chi phí 1, đóng góp 1 , tổng 5
Kết quả: 5


## 9. Code nộp LeetCode

In [18]:
from collections import Counter

class Solution:
    def minimumKeypresses(self, s: str) -> int:
        freq = sorted(Counter(s).values(), reverse=True)   # chữ hay dùng trước
        ans = 0
        for i, v in enumerate(freq):
            ans += v * (i // 9 + 1)                         # 9 ô vị trí 1, rồi 9 ô vị trí 2, ...
        return ans

In [19]:
sol = Solution()
for s, exp in examples:
    got = sol.minimumKeypresses(s)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

5 | kỳ vọng 5 | OK
15 | kỳ vọng 15 | OK


## 10. Vì sao thuật toán đúng

**Mô hình.** Một bàn phím hợp lệ ⇔ một đơn ánh từ 26 chữ vào 27 ô `(nút, vị trí)`, sao cho mỗi nút dùng các vị trí liên tiếp từ 1 (nút có 2 chữ thì chiếm vị trí 1, 2). Chi phí gõ `s` = `Σ_ch f(ch) · vị_trí(ch)`.

**Cận dưới.** Sắp tần suất giảm dần `f₁ ≥ … ≥ f₂₆` (chữ không xuất hiện có `f = 0`). Trong **bất kỳ** bàn phím nào, có tối đa 9 chữ ở vị trí 1 và tối đa 18 chữ ở vị trí ≤ 2. Sắp các chi phí ô mà 26 chữ đang dùng thành dãy tăng `c₁ ≤ c₂ ≤ … ≤ c₂₆`; vì mỗi mức chỉ có 9 ô nên `c_i ≥ ⌈i/9⌉`. Theo bất đẳng thức sắp xếp lại, chi phí của bàn phím ≥ `Σ f_i · c_i` (ghép tần suất lớn với chi phí nhỏ) ≥ `Σ f_i · ⌈i/9⌉`.

**Đạt được.** Bàn phím tham lam ở cách 6 dùng đúng chi phí `⌈i/9⌉` cho chữ hạng `i`, và hợp lệ (mỗi nút nhận chữ theo vòng nên vị trí liên tiếp từ 1, tối đa 3 chữ). ∎

Kiểm chứng thực nghiệm cận dưới: sinh **ngẫu nhiên** hàng nghìn bàn phím hợp lệ (xáo 26 chữ rồi chia vòng tròn vào 9 nút, hoặc chia ngẫu nhiên 26 chữ thành các nút ≤ 3 chữ) và so chi phí với tham lam — không bàn phím nào rẻ hơn.

In [20]:
def random_keypad(rng):
    letters = list(string.ascii_lowercase); rng.shuffle(letters)
    sizes = [3] * 8 + [2]; rng.shuffle(sizes)          # 8 nút 3 chữ + 1 nút 2 chữ = 26
    kp = []; i = 0
    for sz in sizes:
        kp.append(letters[i:i + sz]); i += sz
    return kp

rng10 = random.Random(10)
for _ in range(200):
    text = "".join(rng10.choice(string.ascii_lowercase[:rng10.randint(1, 26)]) for _ in range(rng10.randint(1, 80)))
    best = Solution().minimumKeypresses(text)
    for _ in range(20):
        assert type_cost(random_keypad(rng10), text) >= best
print("4000 bàn phím ngẫu nhiên: không cái nào tốt hơn tham lam")

4000 bàn phím ngẫu nhiên: không cái nào tốt hơn tham lam


## 11. Phân tích Time / Space Complexity

`n = len(s)`, bảng chữ `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị ô | O(27!/(27−d)!) | O(d) |
| 1b. Vét cạn mức chi phí | O(3^d · d) | O(d) |
| 2. Sắp xếp + công thức | O(n + Σ log Σ) | O(Σ) |
| 3. Mảng đếm | O(n + Σ log Σ) | O(Σ) |
| 4. Heap | O(n + Σ log Σ) | O(Σ) |
| 5. Khối 9 | O(n + Σ log Σ) | O(Σ) |

In [21]:
rng = random.Random(2268)
big = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
skew = "".join(rng.choices(string.ascii_lowercase, weights=[2 ** (i % 10) for i in range(26)], k=10**5))
for data, nm in [(big, "đều"), (skew, "lệch")]:
    res = []
    for name, f in [("sort_greedy", sort_greedy), ("count_array", count_array), ("heap_greedy", heap_greedy),
                    ("by_blocks", by_blocks), ("Solution", Solution().minimumKeypresses)]:
        t = time.perf_counter(); res.append(f(data)); dt = time.perf_counter() - t
        print(f"{nm:<5} {name:<12} -> {res[-1]}  {dt*1000:.1f} ms")
    assert len(set(res)) == 1

đều   sort_greedy  -> 195086  2.3 ms
đều   count_array  -> 195086  3.3 ms
đều   heap_greedy  -> 195086  2.7 ms
đều   by_blocks    -> 195086  2.3 ms
đều   Solution     -> 195086  2.8 ms
lệch  sort_greedy  -> 108143  2.8 ms
lệch  count_array  -> 108143  3.7 ms
lệch  heap_greedy  -> 108143  2.2 ms
lệch  by_blocks    -> 108143  2.1 ms
lệch  Solution     -> 108143  2.1 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [22]:
ALL = {"sort_greedy": sort_greedy, "count_array": count_array, "heap_greedy": heap_greedy,
       "by_blocks": by_blocks, "Solution": Solution().minimumKeypresses,
       "keypad": lambda s: type_cost(build_keypad(s), s)}
edge = examples + [
    ("a", 1),
    ("aaaa", 4),
    ("abcdefghi", 9),                        # đúng 9 chữ: tất cả chi phí 1
    ("abcdefghij", 11),                      # chữ thứ 10 chi phí 2
    (string.ascii_lowercase, 9 * 1 + 9 * 2 + 8 * 3),   # 51
    ("z" * 5 + string.ascii_lowercase, None),
]
for s, exp in edge:
    if exp is None: exp = brute_levels(s) if len(set(s)) <= 11 else sort_greedy(s)
    for name, f in ALL.items():
        assert f(s) == exp, (name, s, f(s), exp)
print("Edge case OK; 26 chữ một lần ->", Solution().minimumKeypresses(string.ascii_lowercase))

Edge case OK; 26 chữ một lần -> 51


### 12.2 Đối chiếu ngẫu nhiên với vét cạn mức chi phí (≤ 11 chữ khác nhau)

Vét cạn tốn `3^d`, nên chỉ 20 case dùng tới 10–11 chữ (để thấy chi phí 2), các case còn lại ≤ 7 chữ; thêm 30 case đối chiếu với vét cạn hoán vị ô (≤ 3 chữ).

In [23]:
rng = random.Random(42)
for t in range(150):
    alpha = rng.sample(string.ascii_lowercase, rng.randint(1, 11 if t < 20 else 7))   # vài case > 9 chữ (thấy chi phí 2)
    s = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 40)))
    ref = brute_levels(s)
    for name, f in ALL.items():
        assert f(s) == ref, (name, s)
for _ in range(30):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15)))
    assert brute(s) == Solution().minimumKeypresses(s)
print("đối chiếu vét cạn OK")

đối chiếu vét cạn OK


### 12.3 Tính chất: hoán vị chuỗi không đổi đáp án; thêm một ký tự tăng đáp án đúng bằng chi phí ô mới của nó (≤ 3)

In [24]:
rng = random.Random(7)
for _ in range(500):
    s = "".join(rng.choice(string.ascii_lowercase[:rng.randint(1, 26)]) for _ in range(rng.randint(1, 60)))
    a = Solution().minimumKeypresses(s)
    assert Solution().minimumKeypresses("".join(rng.sample(s, len(s)))) == a
    ch = rng.choice(string.ascii_lowercase)
    d = Solution().minimumKeypresses(s + ch) - a
    assert 1 <= d <= 3
    assert len(s) <= a <= 3 * len(s)
print("tính chất OK")

tính chất OK


### 12.4 Cực trị theo ràng buộc

- `s` dài 10⁵ chỉ một chữ ⇒ đáp án 10⁵ (mọi lần bấm 1).
- `s` dài 10⁵ chia đều 26 chữ ⇒ đáp án `≈ (9·1 + 9·2 + 8·3)/26 · 10⁵ ≈ 1,96·10⁵`.

In [25]:
one = "q" * 10**5
even = (string.ascii_lowercase * (10**5 // 26 + 1))[:10**5]
for name, f in ALL.items():
    assert f(one) == 10**5
print("đều 26 chữ:", Solution().minimumKeypresses(even), "| ước lượng:", round(51 / 26 * 10**5))

đều 26 chữ: 196150 | ước lượng: 196154


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp tần suất **tăng dần**

In [26]:
def wrong_ascending(s):
    vals = sorted(Counter(s).values())                     # SAI: chữ ít dùng nhận ô rẻ
    return sum(v * (i // 9 + 1) for i, v in enumerate(vals))

s = "a" * 50 + "bcdefghijk"
print("bản sai :", wrong_ascending(s), "| bản đúng:", Solution().minimumKeypresses(s))

bản sai : 111 | bản đúng: 62


**Sửa:** `reverse=True` — tần suất lớn nhất ghép chi phí nhỏ nhất.

### Lỗi 2 — Gán theo **thứ tự bảng chữ cái** (bàn phím cố định kiểu điện thoại)

In [27]:
def wrong_alphabetical(s):
    cost = {ch: i // 9 + 1 for i, ch in enumerate(string.ascii_lowercase)}   # SAI: không tối ưu theo s
    return sum(cost[ch] for ch in s)

print("bản sai :", wrong_alphabetical("zzzzz"), "| bản đúng:", Solution().minimumKeypresses("zzzzz"))

bản sai : 15 | bản đúng: 5


**Vì sao sai:** đề cho phép **tự chọn** bàn phím; `z` dùng nhiều thì đặt nó ở vị trí 1.

### Lỗi 3 — Lệch công thức chi phí: `i // 3 + 1` (nhầm 3 chữ mỗi nút)

In [28]:
def wrong_formula(s):
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(v * (i // 3 + 1) for i, v in enumerate(vals))      # SAI

print("bản sai :", wrong_formula("abcdefghijkl"), "| bản đúng:", Solution().minimumKeypresses("abcdefghijkl"))

bản sai : 30 | bản đúng: 15


**Vì sao sai:** chi phí tăng sau mỗi **9** chữ (số nút), không phải mỗi 3 chữ (sức chứa một nút). Hãy hình dung: điền vị trí 1 của **cả 9 nút** trước, rồi mới tới vị trí 2. **Sửa:** `i // 9 + 1`.

### Lỗi 4 — Đếm số chữ **khác nhau** thay vì tần suất

In [29]:
def wrong_distinct_only(s):
    d = len(set(s))
    return sum(i // 9 + 1 for i in range(d))                       # SAI: bỏ qua số lần lặp

print("bản sai :", wrong_distinct_only("apple"), "| bản đúng:", Solution().minimumKeypresses("apple"))

bản sai : 4 | bản đúng: 5


**Sửa:** mỗi lần xuất hiện của chữ đều tốn chi phí ô của nó ⇒ nhân với tần suất.

### Lỗi 5 — Dùng 1-based `i` trong `enumerate` mà vẫn viết `i // 9`

In [30]:
def wrong_index_base(s):
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(v * (i // 9 + 1) for i, v in enumerate(vals, 1))   # SAI: chữ hạng 9 thành chi phí 2

print("bản sai :", wrong_index_base("abcdefghi"), "| bản đúng:", Solution().minimumKeypresses("abcdefghi"))

bản sai : 10 | bản đúng: 9


**Sửa:** với chỉ số 0-based dùng `i // 9 + 1`; với 1-based dùng `(i − 1) // 9 + 1` (hay `ceil(i/9)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tự sắp xếp/gán để tổng (trọng số × chi phí) nhỏ nhất" ⇒ **bất đẳng thức sắp xếp lại**: lớn ghép nhỏ.
- Nhóm "ô" có cùng chi phí ⇒ chỉ cần biết **bao nhiêu** ô mỗi mức (9 ô mỗi mức).
- Chứng minh tham lam bằng **đổi chỗ hai phần tử** — 2 dòng đại số.
- Bảng chữ cái nhỏ (26) ⇒ mảng đếm + sắp xếp 26 phần tử là đủ.
- Bài cùng họ: 1167 Minimum Cost to Connect Sticks, 2285 Maximum Total Importance of Roads (cùng ý "lớn ghép lớn"), 455 Assign Cookies, 1057, 3014/3016 Minimum Number of Pushes to Type Word (bản 8 phím, không giới hạn chữ mỗi phím).

## 15. Biến thể của bài

### 15.1 Biến thể A — `k` nút, mỗi nút tối đa `c` chữ (yêu cầu `k·c ≥ 26`)

In [31]:
def keypresses_general(s, k, c):
    assert k * c >= 26
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(v * (i // k + 1) for i, v in enumerate(vals))

def keypresses_general_brute(s, k, c):
    freq = list(Counter(s).values()); best = None
    for levels in product(range(1, c + 1), repeat=len(freq)):
        if all(levels.count(t) <= k for t in range(1, c + 1)):
            cost = sum(f * t for f, t in zip(freq, levels))
            if best is None or cost < best: best = cost
    return best

print(keypresses_general("abcdefghijkl", 9, 3), keypresses_general("abcdefghijkl", 8, 4), keypresses_general("abcdefghijkl", 26, 1))
rng = random.Random(151)
for _ in range(60):
    k = rng.randint(3, 4); c = -(-26 // k)                  # c = 9 hoặc 7
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 25)))
    assert keypresses_general(s, k, c) == keypresses_general_brute(s, k, c)
print("tổng quát: khớp brute force")

15 16 12


tổng quát: khớp brute force


Giới hạn `c` chữ mỗi nút không ảnh hưởng công thức miễn là `k·c ≥ 26` (đủ chỗ) — ràng buộc thực sự là **k ô mỗi mức chi phí**.

### 15.2 Biến thể B — Chi phí vị trí tùy ý `w[1] ≤ w[2] ≤ …` (ví dụ bấm giữ lâu hơn)

Tham lam vẫn đúng (bất đẳng thức sắp xếp lại chỉ cần chi phí **không giảm** theo vị trí).

In [32]:
def keypresses_weighted(s, w, k=9):
    vals = sorted(Counter(s).values(), reverse=True)
    return sum(v * w[i // k] for i, v in enumerate(vals))

print(keypresses_weighted("abcdefghijkl", [1, 2, 3]), keypresses_weighted("abcdefghijkl", [1, 5, 10]))

15 24


### 15.3 Biến thể C — Thứ tự chữ trên mỗi nút **cố định theo bảng chữ cái**, chỉ được chọn **chia đoạn** (mỗi nút là một đoạn liên tiếp của `a..z`)

Không còn tham lam đơn giản: DP trên (số chữ đã dùng, số nút đã dùng), mỗi nút lấy 1..3 chữ liên tiếp. O(26·9·3).

In [33]:
def keypresses_contiguous(s):
    f = Counter(s); A = string.ascii_lowercase; INF = float("inf")
    dp = [[INF] * 10 for _ in range(27)]; dp[0][0] = 0
    for i in range(26):
        for b in range(9):
            if dp[i][b] == INF: continue
            cost = 0
            for t in range(1, 4):
                if i + t > 26: break
                cost += f[A[i + t - 1]] * t
                dp[i + t][b + 1] = min(dp[i + t][b + 1], dp[i][b] + cost)
    return min(dp[26])

print(keypresses_contiguous("apple"), "(tự do:", Solution().minimumKeypresses("apple"), ")",
      keypresses_contiguous("abcdefghijkl"))

8 (tự do: 5 ) 22


### 15.4 Biến thể D — LC 3016: 8 phím (2..9), **không giới hạn** số chữ mỗi phím

Chi phí tăng sau mỗi 8 chữ: `i // 8 + 1` (có thể tới 4 vì 26 = 8·3 + 2). Cùng tham lam.

In [34]:
def minimum_pushes(word):
    vals = sorted(Counter(word).values(), reverse=True)
    return sum(v * (i // 8 + 1) for i, v in enumerate(vals))

print(minimum_pushes("abcde"), minimum_pushes("xyzxyzxyzxyz"), minimum_pushes("aabbccddeeffgghhiiiiii"))
print("26 chữ một lần:", minimum_pushes(string.ascii_lowercase), "= 8*1 + 8*2 + 8*3 + 2*4 =", 8 + 16 + 24 + 8)

5 12 24
26 chữ một lần: 56 = 8*1 + 8*2 + 8*3 + 2*4 = 56


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị ô | thử mọi bàn phím | mũ | O(d) | kiểm chứng rất nhỏ |
| 1b. Vét cạn mức | mỗi chữ chọn chi phí 1/2/3 | O(3^d) | O(d) | kiểm chứng |
| 2. Sắp xếp + công thức | `v · (i//9 + 1)` | O(n) | O(26) | **mặc định** |
| 3. Mảng đếm | không Counter | O(n) | O(26) | ngôn ngữ khác |
| 4. Heap | lấy lớn nhất dần | O(n) | O(26) | streaming |
| 5. Khối 9 | tổng từng khối × chi phí | O(n) | O(26) | dễ đọc |
| 6. Dựng bàn phím | gán vòng tròn 9 nút | O(n) | O(26) | cần xuất bàn phím |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumKeypresses("aaaaaaaaaabcdefghijk")` (10 chữ `a` + `b..k`)?

**Bài 2 (tính tay):** Với 26 chữ, chữ thứ `i` (theo bảng chữ cái) xuất hiện `i` lần (`a` 1 lần … `z` 26 lần). Tính đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(s)` trả về **bàn phím** tối ưu dạng list 9 chuỗi (mỗi chuỗi ≤ 3 chữ, đủ 26 chữ).

**Bài 4 (tự viết lại):** Viết `bt4_todo(s)` dùng mảng đếm 26 phần tử + sắp xếp, không dùng `Counter`.

**Bài 5 (mini challenge):** Có `q` truy vấn `(l, r)`: đáp án của bài gốc trên chuỗi con `s[l..r]`. Viết `bt5_todo(s, queries)` dùng **tiền tố đếm theo từng chữ** (26 mảng tiền tố), O(26 log 26) mỗi truy vấn.

In [35]:
def bt3_todo(s):
    # TODO: bàn phím tối ưu (list 9 chuỗi)
    pass

def bt4_todo(s):
    # TODO: mảng đếm 26
    pass

def bt5_todo(s, queries):
    # TODO: truy vấn trên chuỗi con bằng tiền tố đếm
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tần suất: a:10, 10 chữ khác 1 lần. Mức 2: hạng 1–9: `a` + 8 chữ ⇒ `10 + 8 = 18`; hạng 10–11: 2 chữ × 2 = 4. Mức 3: `22`.

**Bài 2.** Mức 1: tần suất giảm dần `26, 25, …, 1`. Mức 2: khối 1 = 26..18 (tổng 198), khối 2 = 17..9 (117), khối 3 = 8..1 (36). Mức 3: `198·1 + 117·2 + 36·3 = 540`.

**Bài 3.** Mức 1: thứ tự chữ theo tần suất giảm, thêm chữ không dùng. Mức 2: chữ hạng `i` vào nút `i % 9`. Mức 3: `"".join` từng nút.

**Bài 4.** Mức 1: `cnt[ord(ch) − 97] += 1`. Mức 2: `cnt.sort(reverse=True)`. Mức 3: dừng khi gặp 0.

**Bài 5.** Mức 1: `pre[c][i]` = số lần chữ `c` trong `s[:i]`. Mức 2: tần suất chữ `c` trong `s[l..r]` = `pre[c][r+1] − pre[c][l]`. Mức 3: sắp 26 tần suất, áp công thức.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `22`. **Bài 2:** `540`.

In [36]:
assert Solution().minimumKeypresses("a" * 10 + "bcdefghijk") == 22
s2 = "".join(ch * (i + 1) for i, ch in enumerate(string.ascii_lowercase))
assert Solution().minimumKeypresses(s2) == 540

def bt3(s):
    return ["".join(b) for b in build_keypad(s)]

def bt4(s):
    return count_array(s)

def bt5(s, queries):
    n = len(s); pre = [[0] * (n + 1) for _ in range(26)]
    for i, ch in enumerate(s):
        c = ord(ch) - 97
        for t in range(26): pre[t][i + 1] = pre[t][i] + (t == c)
    out = []
    for l, r in queries:
        vals = sorted((pre[t][r + 1] - pre[t][l] for t in range(26)), reverse=True)
        out.append(sum(v * (i // 9 + 1) for i, v in enumerate(vals)))
    return out

rng = random.Random(1968)
for _ in range(200):
    s = "".join(rng.choice(string.ascii_lowercase[:rng.randint(1, 26)]) for _ in range(rng.randint(1, 50)))
    kp = bt3(s)
    assert len(kp) == 9 and all(len(b) <= 3 for b in kp) and sorted("".join(kp)) == list(string.ascii_lowercase)
    pos = {ch: t + 1 for b in kp for t, ch in enumerate(b)}
    assert sum(pos[ch] for ch in s) == Solution().minimumKeypresses(s)
    assert bt4(s) == Solution().minimumKeypresses(s)
    qs = []
    for _ in range(3):
        l = rng.randrange(len(s)); r = rng.randrange(l, len(s)); qs.append((l, r))
    assert bt5(s, qs) == [Solution().minimumKeypresses(s[l:r + 1]) for l, r in qs]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
freq = sorted(Counter(s).values(), reverse=True)     # lớn trước
return sum(v * (i // 9 + 1) for i, v in enumerate(freq))   # 9 ô mỗi mức chi phí
```

**Checklist:**
- [ ] Sắp tần suất **giảm dần**.
- [ ] Chi phí tăng sau mỗi **9** chữ (số nút), dùng chỉ số 0-based.
- [ ] Nhân với **tần suất**, không chỉ đếm chữ khác nhau.
- [ ] Không dùng bảng chữ cái cố định — bàn phím do mình chọn.
- [ ] Đã thử: một chữ, đúng 9 chữ, đúng 10 chữ, đủ 26 chữ.
- [ ] (Premium) Đối chiếu lại đề gốc.